# Final results

- the final calculations in one place: RMSE, MAE, directional accuracy, and weighted directional accuracy (the profitability test)
- reads the forecast tables final2.ipynb saves (holdout_forecasts.csv and forward_forecasts.csv), so run final2 first
- holdout = 1100+ windows from 2021-09 onward, forward = true out-of-sample windows after 2026-05-14 on the frozen pipeline
- directional accuracy skips unwinnable ties and shows the base rate (random walk shows nan, it predicts zero change)
- weighted DA: trade every EWA predicted move of at least 1bp, 0.5bp cost per trade, capture = bp won / bp available, weighted DA = (1 + capture) / 2

In [1]:
import pandas as pd
import numpy as np

holdout = pd.read_csv("../data/generated/final2_results/holdout_forecasts.csv", parse_dates=["origin_date", "target_date"])
forward = pd.read_csv("../data/generated/final2_results/forward_forecasts.csv", parse_dates=["origin_date", "target_date"])

horizons = [1, 5, 20]
modelNames = ["Random Walk", "VARX", "EWA Ensemble"]
thresholdBp = 1.0
costBp = 0.5

print("holdout windows: ", holdout["origin_date"].nunique(), " (", holdout["origin_date"].min().date(), " to ", holdout["origin_date"].max().date(), ")")
print("forward windows: ", forward["origin_date"].nunique(), " (", forward["origin_date"].min().date(), " to ", forward["origin_date"].max().date(), ")")

holdout windows:  1149  ( 2021-09-09  to  2026-04-16 )
forward windows:  19  ( 2026-05-15  to  2026-06-11 )


#### Error calculations

- one block per dataset and model, per horizon lines in the baseline format
- then the weighted directional accuracy of the EWA Ensemble for each dataset

In [2]:
def printMetrics(table, datasetName):
    print("##############################")
    print("Dataset: ", datasetName)
    print("##############################")

    for modelName in modelNames:
        print("==============================")
        print("Model: ", modelName)
        print("==============================")
        modelRows = table[table["model"] == modelName]

        for horizon in horizons:
            rows = modelRows[modelRows["horizon"] == horizon]
            errors = 100 * (rows["predicted_yield"] - rows["actual_yield"]).to_numpy()
            rmse = np.sqrt(np.mean(errors ** 2))
            mae = np.mean(np.abs(errors))

            predictedSign = np.sign((rows["predicted_yield"] - rows["origin_yield"]).to_numpy())
            actualSign = np.sign((rows["actual_yield"] - rows["origin_yield"]).to_numpy())
            #scorable rows need both signs nonzero: zero predictions have no direction,
            #and exact-zero realized changes are unwinnable ties
            scorable = (predictedSign != 0) & (actualSign != 0)
            if scorable.sum() > 0:
                directionalAccuracy = 100 * np.mean(predictedSign[scorable] == actualSign[scorable])
            else:
                directionalAccuracy = np.nan

            nonzero = actualSign[actualSign != 0]
            baseRate = 100 * max(np.mean(nonzero > 0), np.mean(nonzero < 0))

            print(f"Horizon: {horizon} day, rmse: {rmse}, MAE: {mae}, Directional Accuracy: {directionalAccuracy} %, base rate: {baseRate} % ")
        print()

    #weighted directional accuracy of the EWA Ensemble (the profitability test)
    print("weighted directional accuracy, EWA Ensemble (", thresholdBp, "bp threshold, ", costBp, "bp cost per trade)")
    ewaRows = table[table["model"] == "EWA Ensemble"]
    for horizon in horizons:
        rows = ewaRows[ewaRows["horizon"] == horizon]
        predictedMove = 100 * (rows["predicted_yield"] - rows["origin_yield"]).to_numpy()
        actualMove = 100 * (rows["actual_yield"] - rows["origin_yield"]).to_numpy()

        pnl = 0.0
        available = 0.0
        trades = 0
        for t in range(len(predictedMove)):
            if abs(predictedMove[t]) >= thresholdBp:
                pnl = pnl + np.sign(predictedMove[t]) * actualMove[t] - costBp
                available = available + abs(actualMove[t])
                trades = trades + 1

        if available > 0:
            capture = pnl / available
            weightedDa = (1.0 + capture) / 2.0
            print(f"Horizon: {horizon} day, weighted DA: {round(weightedDa, 4)}, capture: {round(capture, 4)}, trades: {trades}, net pnl: {round(pnl, 1)} bp")
        else:
            print(f"Horizon: {horizon} day, no trades cleared the threshold")
    print()


printMetrics(holdout, "holdout (2021-09 onward)")
printMetrics(forward, "forward test (after 2026-05-14, frozen pipeline)")

##############################
Dataset:  holdout (2021-09 onward)
##############################
Model:  Random Walk
Horizon: 1 day, rmse: 6.111764656832587, MAE: 4.145343777197563, Directional Accuracy: nan %, base rate: 51.37664207306101 % 
Horizon: 5 day, rmse: 13.194960041173733, MAE: 9.1994619827518, Directional Accuracy: nan %, base rate: 55.74819054031308 % 
Horizon: 20 day, rmse: 28.028830115927725, MAE: 20.43674341324472, Directional Accuracy: nan %, base rate: 57.039505771419286 % 

Model:  VARX
Horizon: 1 day, rmse: 6.115001736331455, MAE: 4.176150198076738, Directional Accuracy: 52.82526543098795 %, base rate: 51.37664207306101 % 
Horizon: 5 day, rmse: 13.092947022264083, MAE: 9.17592679234941, Directional Accuracy: 56.3457330415755 %, base rate: 55.74819054031308 % 
Horizon: 20 day, rmse: 27.035527572421802, MAE: 19.78421597115533, Directional Accuracy: 58.29946350186962 %, base rate: 57.039505771419286 % 

Model:  EWA Ensemble
Horizon: 1 day, rmse: 6.105249345806767, MAE: